# Vietnamese Traffic Violation Detection Demo

This notebook demonstrates the project pipeline for single-image traffic violation detection:

1. Object detection extracts vehicles, traffic lights, stop lines, and road objects.
2. Scene geometry converts detections into structured facts.
3. The rule engine creates candidate violations.
4. Violated-object segmentation exports crops, masks, and overlays.
5. Qwen2.5-VL can optionally verify/explain the final violation.

The first demo uses a fake detector so the notebook can run without downloading heavy ML models. The real detector and Qwen cells are optional.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

IMAGE_PATH = PROJECT_ROOT / 'dataset/image/parking/violate/2.png'
SCENE_CONFIG = PROJECT_ROOT / 'configs/scene_config.example.json'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/notebook_demo'
SEGMENT_DIR = OUTPUT_DIR / 'segments'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Image exists:', IMAGE_PATH.exists(), IMAGE_PATH)
print('Scene config exists:', SCENE_CONFIG.exists(), SCENE_CONFIG)

## Environment Check

The full pipeline needs `torch`, `transformers`, `accelerate`, `qwen-vl-utils`, and `Pillow`. If these are missing, the fake-detector demo below still works.

In [ ]:
for module_name in ['torch', 'transformers', 'accelerate', 'qwen_vl_utils', 'PIL']:
    try:
        module = __import__(module_name)
        print(f'{module_name}: installed {getattr(module, "__version__", "unknown")}')
    except Exception as exc:
        print(f'{module_name}: missing/error: {exc}')

## Lightweight Demo With Fake Detector

This cell simulates detector output. It lets us test the rule engine, segmentation export, VLM prompt creation, and RAG query generation without loading GroundingDINO or Qwen.

In [ ]:
from illegal_detector.traffic_violation_pipeline import TrafficViolationPipeline

class FakeTensor:
    def __init__(self, value):
        self.value = value

    def item(self):
        return self.value

class FakeBox(list):
    def tolist(self):
        return list(self)

class FakeDetector:
    def detect(self, image_path, text_labels, threshold=0.35, text_threshold=0.25):
        return [], {
            'boxes': [
                FakeBox([300, 500, 420, 700]),
                FakeBox([650, 120, 700, 220]),
            ],
            'scores': [FakeTensor(0.91), FakeTensor(0.88)],
            'labels': ['motorbike', 'red traffic light'],
        }

    def draw_boxes(self, image_path, output_path=None, show=False):
        return None

pipeline = TrafficViolationPipeline(detector=FakeDetector())
result = pipeline.analyze(
    image_path=str(IMAGE_PATH),
    scene_config_path=str(SCENE_CONFIG),
    segment_dir=str(SEGMENT_DIR),
)

print(json.dumps({
    'facts': result['facts'],
    'violations': result['violations'],
    'rag_queries': result['rag_queries'],
    'violated_object_segments': result['violated_object_segments'],
}, indent=2, ensure_ascii=False))

## View Segmentation Outputs

The default segmenter creates bounding-box masks for vehicles involved in violation candidates. This is not SAM/SAM2 pixel-accurate segmentation yet, but it gives visual evidence assets for reports and VLM review.

In [ ]:
from IPython.display import display
from PIL import Image

for segment in result['violated_object_segments'][:2]:
    print(segment['violation_type'], segment['vehicle_id'])
    display(Image.open(segment['crop_path']))
    display(Image.open(segment['overlay_path']))

## Inspect The VLM Prompt

The prompt contains the image path, detector/geometry facts, and rule-engine candidates. When Qwen2.5-VL is enabled, the image and this prompt are sent together.

In [ ]:
print(result['vlm_prompt'][:2500])

## Real Pipeline Command

Run this after installing dependencies with `pip install -r requirements.txt`. This uses GroundingDINO for detection and saves JSON, annotations, and violated-object segments.

In [ ]:
cmd = f"""
python3 {PROJECT_ROOT / 'main.py'} {IMAGE_PATH} \
  --scene-config {SCENE_CONFIG} \
  --output {OUTPUT_DIR / 'real_result.json'} \
  --annotate {OUTPUT_DIR / 'real_annotated.png'} \
  --segment-dir {OUTPUT_DIR / 'real_segments'}
""".strip()
print(cmd)
# Uncomment to run after dependencies are installed:
# !{cmd}

## Real Pipeline With Qwen2.5-VL

This runs the detector, rule engine, segmentation, and Qwen2.5-VL. On CPU, use the 3B model and keep generated tokens low. It may still be slow.

In [ ]:
qwen_cmd = f"""
python3 {PROJECT_ROOT / 'main.py'} {IMAGE_PATH} \
  --scene-config {SCENE_CONFIG} \
  --output {OUTPUT_DIR / 'qwen_result.json'} \
  --annotate {OUTPUT_DIR / 'qwen_annotated.png'} \
  --segment-dir {OUTPUT_DIR / 'qwen_segments'} \
  --vlm qwen \
  --qwen-model Qwen/Qwen2.5-VL-3B-Instruct \
  --vlm-max-new-tokens 256
""".strip()
print(qwen_cmd)
# Uncomment to run after dependencies are installed:
# !{qwen_cmd}